# Validation: Dataset Generation

### Dataset 1: data_recon
---
### 1. Perovskite (Unit Cell)
- Metrics: reconstruction strcuture
- Note: 5 atoms per structure
- Note: manually created pseudo-structure

### 2. Perovskite (Super Cell)
- Metrics: reconstruction strcuture
- Note: 40 atoms per structure

### Dataset 2: data_inpaint
---
### 1. Inpainting on LaCl3-like Materials
- Metrics: reconstruction strcuture
- Note:  LaCl3 (unit cell), LaCl3 (super cell), Nb/LaCl3 (unit cell), 

### 2. Inpainting on Li-O Containing Compounds
- Metrics: reconstruction strcuture

In [53]:
import sys
sys.path.append('../')

import pandas as pd

import torch

from pymatgen.core import Structure
from pymatgen.io.cif import CifWriter
from pymatgen.transformations.standard_transformations import OrderDisorderedStructureTransformation

from chggen.common.data_utils import get_pymatgen_structure

device = 'cuda:6'

### 01 Generate Dataset for Reconstruction 

In [45]:
cur_frac_coords = torch.rand(50, 3, requires_grad = False, device = device)
cur_atom_types = torch.tensor([7, 7, 8, 22, 20, 7, 8, 8, 22, 20,7, 7, 8, 40, 20,7, 7, 8, 72, 20,7, 7, 8, 22, 20
                               ,7, 7, 8, 22, 20,7, 7, 8, 28, 30,7, 7, 8, 40, 56,7, 7, 8, 22, 20,8, 8, 8, 40, 20], device = device)
num_atoms = torch.tensor([5,5,5,5,5,5,5,5,5,5], device = device)

length = 4
angle = 90
lengths = torch.ones((10,3), device=device) * length
angles = torch.ones((10,3), device = device) * 90

In [51]:
# Initialize pd frame
df = pd.DataFrame(columns=['cif'])

s_list = get_pymatgen_structure(
    lengths = lengths,         
    angles = angles,
    num_atoms = num_atoms,
    frac_coords = cur_frac_coords,
    atom_types = cur_atom_types,
)

# Convert structures to cif text with CifWriter.
cifs = []
for s in s_list:
    cifs.append(CifWriter(s).__str__()) 
        
# Supercell 2*2*2
s_list = [s * (2, 2, 2) for s in s_list]
for s in s_list:
    cifs.append(CifWriter(s).__str__()) 

df['cif'] = cifs

In [49]:
# Save to csv
df.to_csv('../data/dataset/data_recon.csv', index=False)

### 02 Generate Dataset for Inpainting

In [97]:
# Initialize pd frame
df = pd.DataFrame(columns=['cif', 'num_insert_ion'])

s_list = []
num_insert_ions = []

# LaCl3 unit cell
s0 = Structure.from_file('../../gen_org/LaCl3.cif')
s_list.append(s0.copy())
num_insert_ions.append(1)

# LaCl3 super cell
s0 *= [2, 2, 2]
s_list.append(s0.copy())
num_insert_ions.append(8)

# LaCl3 disordered (La / Nb)
s0.replace_species({'La3+': {'Nb5+':0.5, 'La3+': 0.5}})
order = OrderDisorderedStructureTransformation(algo = 2)
s0 = order.apply_transformation(s0, return_ranked_list = False)
s_list.append(s0.copy())
num_insert_ions.append(8)

# Li-O 
mp_O_Li = pd.read_csv('../../lithium/data/mp_O_Li.csv', keep_default_na=False, na_values=[''])
for num in range(10):
    s_list.append(Structure.from_str(mp_O_Li['cif'][num], fmt='cif', frac_tolerance=0, site_tolerance=0))
    num_insert_ions.append(int(s_list[-1].composition.get_el_amt_dict()['Li']))

cifs = []
for s in s_list:
    cifs.append(CifWriter(s).__str__())
    
df['cif'] = cifs
df['num_insert_ion'] = num_insert_ions

/home/xinzhedai/miniconda3/envs/chggen/lib/python3.8/site-packages/pymatgen/io/cif.py:1134: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [98]:
# Save to csv
df.to_csv('../data/dataset/data_inpaint.csv', index=False)